In [16]:
import numpy as np
import scipy
import scipy.integrate as integrate
import matplotlib.pyplot as plt
from scipy.special import gamma as gamma
from scipy.optimize import root_scalar

## Distribución Gaussiana (Normal)

In [3]:
def gauss(mu, sigma, x):
    return 1.0 / (np.sqrt(2.0 * np.pi) * sigma) * np.exp(-((x - mu) ** 2) / (2.0 * sigma ** 2))

# Ejemplo de uso para graficar y resolver la integral
# mu1 = 2.0; sigma1 = 2.0
# x_arr = np.linspace(-10, 15, 500)
# pg1 = gauss(mu1, sigma1, x_arr)
# plt.plot(x_arr, pg1, linestyle='-', color='red')

### Problema 1
(a) ¿Qu´e resultado obtiene si n1 = n2 = 400, μ1 = μ2, y σ1 = 20 y σ2 = 30?

In [4]:
def gauss(x, mu, sigma):
    return 1.0 / (np.sqrt(2.0 * np.pi) * sigma) * np.exp(-((x - mu) ** 2) / (2.0 * sigma ** 2))

# 2. Parámetros del Problema 1
mu_diff = 0.0 
sigma_diff = np.sqrt((20**2 / 400) + (30**2 / 400)) # Esto da aprox 1.8027

# 3. Definir la probabilidad que te pide el problema 
# (Supongamos que el problema te pedía un 99% de probabilidad en el centro)
probabilidad_objetivo = 0.99

# 4. Crear un arreglo de posibles valores para buscar el límite. 
# Buscamos desde 0 hasta 10 (un rango lo suficientemente amplio).
posibles_limites = np.linspace(0, 10, 5000)
limite_superior = 0.0

# 5. Iterar hasta encontrar el área correcta
for k in posibles_limites:
    # Integramos desde -k hasta k
    prob, _ = integrate.quad(gauss, -k, k, args=(mu_diff, sigma_diff))
    
    # Si el área calculada alcanza o supera nuestra probabilidad objetivo, nos detenemos
    if prob >= probabilidad_objetivo:
        limite_superior = k
        break

# Como la distribución normal es simétrica (centrada en 0), el límite inferior es negativo
limite_inferior = -limite_superior

print(f"Límite inferior: {limite_inferior:.4f}")
print(f"Límite superior: {limite_superior:.4f}")

Límite inferior: -4.6449
Límite superior: 4.6449


### Problema 2
Se toman muestras aleatorias independientes de tama˜no n1 = 30 y n2 = 50 de dos poblaciones Gaussianas
con medias μ1 = 78 y μ2 = 75 y varianzas σ2
1 = 78 y σ2
2 = 200. Encuentre la probabilidad de que la
media de la primera muestra exceder´a la de la segunda muestra por al menos 4.8.

In [5]:
def gauss(x, mu, sigma):
    return 1.0 / (np.sqrt(2.0 * np.pi) * sigma) * np.exp(-((x - mu) ** 2) / (2.0 * sigma ** 2))

# 2. Definir los parámetros extraídos de la integral
mu = 3.0
sigma = np.sqrt(6.6)

# 3. Limites de integración
limite_inferior = 4.8
limite_superior = np.inf

# 4. Calcular la integral
probabilidad, _ = integrate.quad(gauss, limite_inferior, limite_superior, args=(mu, sigma))

print(f"La probabilidad es: {probabilidad:.4f}")

La probabilidad es: 0.2418


### Problema 3
En un proceso qu´ımico, la cantidad de cierto tipo de impurezas en el producto es dif´ıcil de controlar y
por ello es una variable aleatoria. Se especula que la cantidad media de la poblaci´on de impurezas es
0.20 gramos por gramo de producto. Se sabe que la desviaci´on est´andar es 0.1 gramos por por gramo. Se
realiza un experimento para aprender m´as con respecto a la especulaci´on de que μ = 0.2. El proceso se
lleva a cabo 50 veces en un laboratorio y el promedio de la muestra X resulta ser 0.23 gramos por gramo.
Comente, con resultados cuantitativos, sobre la especulaci´on de que la cantidad media de impurezas es
0.20 gramos por gramo.

In [6]:
# 1. Tu función gaussiana corregida ('x' al inicio)
def gauss(x, mu, sigma):
    return 1.0 / (np.sqrt(2.0 * np.pi) * sigma) * np.exp(-((x - mu) ** 2) / (2.0 * sigma ** 2))

# 2. Definir los parámetros del Problema 3
mu = 0.20
sigma_poblacional = 0.1
n = 50

# Calculamos la desviación estándar de la media muestral (sigma / raíz de n)
sigma_muestral = sigma_poblacional / np.sqrt(n)

# 3. Límites de integración
limite_inferior = 0.23
limite_superior = np.inf

# 4. Calcular la integral (esto nos da la probabilidad de la cola derecha)
prob_una_cola, _ = integrate.quad(gauss, limite_inferior, limite_superior, args=(mu, sigma_muestral))

# 5. Calcular la probabilidad a dos colas (el p-value total)
prob_dos_colas = 2 * prob_una_cola

print(f"Probabilidad de una cola (>= 0.23): {prob_una_cola:.4f}")
print(f"Probabilidad de dos colas (p-value): {prob_dos_colas:.4f}")

Probabilidad de una cola (>= 0.23): 0.0169
Probabilidad de dos colas (p-value): 0.0339


## Distribución t de Student

In [7]:
def tdist(t, nu):
    return (gamma((nu + 1) * 0.5) / (gamma(nu * 0.5) * np.sqrt(np.pi * nu))) * (1.0 + t**2 / nu) ** (-(nu + 1.0) * 0.5)

# Calculadora de valor crítico (ejemplo con probabilidad >= 0.975 y nu=19)
nu = 19
t_arr = np.linspace(-10, 10, 1000)
ss = 0

for tval in t_arr:
    prob = integrate.quad(tdist, -np.inf, tval, args=(nu))
    if prob[0] >= 0.975:
        t_crit = t_arr[ss - 1]
        print("Valor crítico de t:", t_crit)
        break
    ss += 1

# Comprobación de normalización
# integrate.quad(tdist, -np.inf, np.inf, args=(nu))

Valor crítico de t: 2.0920920920920913


### Problema 4
•Encuentre t0.025 con v = 14.
• Encuentre −t0.10 con v = 10.
• Encuentre t0.995 con v = 7.

In [28]:
def dist_t(t, nu):
    return (gamma((nu + 1) * 0.5) / (np.sqrt(np.pi * nu) * gamma(nu * 0.5))) * (1.0 + (t**2)/nu)**(-(nu + 1) * 0.5)

# 2. Definición de la función de búsqueda que te faltaba
def buscar_t(nu, alfa, cola_superior=True):
    # Definimos t_arr dentro de la función para tener el rango de búsqueda
    t_arr = np.linspace(-5, 5, 5000) 
    
    for t_val in t_arr:
        if cola_superior:
            prob, _ = integrate.quad(dist_t, t_val, np.inf, args=(nu,))
            if prob <= alfa: return t_val
        else:
            prob, _ = integrate.quad(dist_t, -np.inf, t_val, args=(nu,))
            if prob >= alfa: return t_val
    return None

# 3. Impresión de resultados
print(f"Problema 4a (t_0.025, v=14): {buscar_t(14, 0.025):.3f}")
print(f"Problema 4b (-t_0.10, v=10): {-buscar_t(10, 0.10):.3f}") # Simétrico
print(f"Problema 4c (t_0.995, v=7): {buscar_t(7, 0.995):.3f}") # Cola gigante

Problema 4a (t_0.025, v=14): 2.145
Problema 4b (-t_0.10, v=10): -1.373
Problema 4c (t_0.995, v=7): -3.498


### Problema 5
Una empresa manufacturera afirma que las bater´ıas que utiliza en sus juegos electr´onicos duran un
promedio de 30 horas. Para mantener este promedio, se prueban 16 bater´ıas cada mes. Si el valor de t
que se calcula cae entre −t0.025 y t0.025, la empresa queda satisfecha con su afirmaci´on. ¿Qu´e conclusiones
deber´ıa obtener la empresa de una muestra que tiene una media de X y una desviaci´on est´andar de S = 5
horas? Suponga que la distribuci´on de las duraciones de las bater´ıas es aproximadamente normal.

In [10]:
# 1. Definir la función t de Student para v = 15
def dist_t_15(t):
    nu = 15
    coef = gamma((nu + 1) * 0.5) / (np.sqrt(np.pi * nu) * gamma(nu * 0.5))
    return coef * (1.0 + (t**2)/nu)**(-(nu + 1) * 0.5)

# 2. Búsqueda iterativa del valor crítico t_0.025
t_arr = np.linspace(0, 5, 5000)
t_critico = 0.0

for t_val in t_arr:
    prob, _ = integrate.quad(dist_t_15, t_val, np.inf)
    if prob <= 0.025:
        t_critico = t_val
        break

print(f"Límites de aceptación: entre {-t_critico:.3f} y {t_critico:.3f}")

# 3. Calcular el estadístico t de la muestra
X_barra = 27.5 # <-- Valor propuesto para hacer el ejercicio

mu = 30.0
S = 5.0
n = 16

t_calculado = (X_barra - mu) / (S / np.sqrt(n))
print(f"Valor t calculado: {t_calculado:.3f}")

# 4. Evaluación y Conclusión Final
if -t_critico <= t_calculado <= t_critico:
    print("Conclusión: El estadístico cae DENTRO del intervalo.")
    print("La empresa queda SATISFECHA con su afirmación de 30 horas.")
else:
    print("Conclusión: El estadístico cae FUERA del intervalo.")
    print("La empresa NO DEBE estar satisfecha con la afirmación.")

Límites de aceptación: entre -2.132 y 2.132
Valor t calculado: -2.000
Conclusión: El estadístico cae DENTRO del intervalo.
La empresa queda SATISFECHA con su afirmación de 30 horas.


### Problema 11
Se asegura que una nueva dieta reduce el peso de la persona por 4.5kg en promedio en un per´ıodo de
dos semanas. Los pesos de 7 mujeres que siguen esta dieta se registran en la tabla subsecuente, antes y
despu´es del per´ıodo de 2 semanas:
Pruebe la aseveraci´on sobre la dieta calculando el intervalo de confianza del 95%para la diferencia del
promedio de los peos. Asuma que las diferencias en pesos son aproximadamente distribuidos normalmente.
(a) ¿Qu´e tipo de observaciones corresponden a los datos registrados en la tabla?

In [24]:
def tdist(t, nu):
    coef = gamma((nu + 1) * 0.5) / (gamma(nu * 0.5) * np.sqrt(np.pi * nu))
    return coef * (1.0 + (t**2) / nu) ** (-(nu + 1.0) * 0.5)

# 2. Datos del problema 11 (Pesos antes y después)
peso_antes = np.array([58.5, 60.3, 61.7, 69.0, 64.0, 62.6, 56.7])
peso_despues = np.array([60.0, 54.9, 58.1, 62.1, 58.5, 59.9, 54.4])

# Calcular diferencias (Antes - Después)
diferencias = peso_antes - peso_despues

# 3. Estadísticos manuales (Media y Desviación Estándar)
n = np.size(diferencias)
nu = n - 1
media_d = np.mean(diferencias)

# Varianza y desviación estándar muestral manual (para no usar librerías extra)
sumn = 0.0
for element in diferencias:
    sumn += (element - media_d) ** 2
var_d = sumn / (n - 1.0)
std_d = np.sqrt(var_d)

# 4. Encontrar el valor crítico t_0.025 mediante integración y root_scalar
alpha_mitad = 0.025

def objetivo_t(t_val):
    # Calcula el área desde t_val hasta infinito y le resta 0.025
    area, _ = integrate.quad(tdist, t_val, np.inf, args=(nu,))
    return area - alpha_mitad
    
# root_scalar encuentra el valor exacto donde el área es igual a 0.025
res = root_scalar(objetivo_t, bracket=[0.0, 10.0], method='brentq')
t_critico = res.root

# 5. Calcular el intervalo de confianza
margen_error = t_critico * (std_d / np.sqrt(n))
limite_inferior = media_d - margen_error
limite_superior = media_d + margen_error

print(f"Media de las diferencias: {media_d:.3f} kg")
print(f"Desviación estándar de las diferencias: {std_d:.3f} kg")
print(f"Valor crítico t (v={nu}): {t_critico:.3f}")
print(f"Intervalo de confianza al 95%: [{limite_inferior:.2f}, {limite_superior:.2f}] kg")

# 6. Evaluación
aseveracion = 4.5
if limite_inferior <= aseveracion <= limite_superior:
    print(f"Conclusión: El valor de {aseveracion} kg ESTÁ dentro del intervalo. La aseveración es plausible.")
else:
    print(f"Conclusión: El valor de {aseveracion} kg NO ESTÁ dentro del intervalo. Se rechaza la aseveración.")

Media de las diferencias: 3.557 kg
Desviación estándar de las diferencias: 2.776 kg
Valor crítico t (v=6): 2.447
Intervalo de confianza al 95%: [0.99, 6.12] kg
Conclusión: El valor de 4.5 kg ESTÁ dentro del intervalo. La aseveración es plausible.


## Distribución Chi-Cuadrada

In [11]:
def chi2(x, nu):
    return 1.0 / (2.0**(nu*0.5) * scipy.special.gamma(nu*0.5)) * x**(nu*0.5 - 1.0) * np.exp(-x*0.5)

def var_muestral(x, x_mean):
    sumn = 0.0
    for element in x:
        sumn += (element - x_mean) ** 2
    return sumn / (np.size(x) - 1.0)

# Ejemplo de cálculo de integral y normalización
nu = 4.0
integral = integrate.quad(chi2, 0.000001, np.inf, args=(nu))
print('nu = %.1f, normalización =' % nu, integral[0])

# Ejemplo del problema de varianza muestral
# x_datos = np.asarray([1.9, 2.4, 3.0, 3.5, 4.2])
# x_mean = np.mean(x_datos)
# s2 = var_muestral(x_datos, x_mean)
# print('La varianza muestral es = ', s2)

nu = 4.0, normalización = 0.999999999999875


### Problema 6
Supongamos que el espesor de una parte usada en un semiconductor es su dimensi´on cr´ıtica y que el
proceso de fabricar estas partes se considera que est´a bajo control si la variaci´on real entre el espesor de
las partes est´a dada por una desviaci´on est´andar no mayor que σ = 0.60 mil´esimas de pulgada. Para
mantener un control sobre el proceso, peri´odicamente se toman muestras aleatorias de tama˜no 20 y se
considera que est´a “fuera de control” si la probabilidad de que S2 ausmir´a un valor mayor que, o igual,
al valor observadp de la muestra es 0.01 o menor (aun cuando σ = 0.60). ¿Qu´e puede uno concluir
sobre el proceso si la desviaci´on est´andar de una muestra aleatoria peri´odica tal es S = 0.84 mil´esimas
de pulgada?

In [14]:
def dist_chi2(x, nu):
    if x <= 0: return 0
    return (1.0 / (2**(nu * 0.5) * gamma(nu * 0.5))) * (x**((nu * 0.5) - 1)) * np.exp(-x * 0.5)
chi_val = (19 * 0.84**2) / (0.60**2)
prob6, _ = integrate.quad(dist_chi2, chi_val, np.inf, args=(19,))
print(f"Problema 6 (Probabilidad fuera de control): {prob6:.4f} (Como es < 0.01, ESTA FUERA DE CONTROL)")

Problema 6 (Probabilidad fuera de control): 0.0074 (Como es < 0.01, ESTA FUERA DE CONTROL)


### Problema 8
Integre la densidad chi cuadrada apropiada para encontrar la probabilidad de que la varianza de una
muestra aleatoria de tma˜no 5 de una poblaci´on normal con σ = 25 caer´a entre 20 y 30.


In [22]:
def dist_chi2(x, nu):
    if x <= 0: return 0
    return (1.0 / (2**(nu * 0.5) * gamma(nu * 0.5))) * (x**((nu * 0.5) - 1)) * np.exp(-x * 0.5)
# Limite inf = (4 * 20)/625, Limite sup = (4 * 30)/625
lim_inf = (4 * 20) / 625
lim_sup = (4 * 30) / 625
prob8, _ = integrate.quad(dist_chi2, lim_inf, lim_sup, args=(4,))
print(f"Problema 8 (Probabilidad entre varianzas): {prob8:.6f}")

Problema 8 (Probabilidad entre varianzas): 0.002361


## Distribución F de Fisher

In [19]:
def Fdist(f, nu1, nu2):
  coef = (gamma((nu1 + nu2) * 0.5)
      * ((nu1 / nu2) ** (nu1 * 0.5))
      / (gamma(nu1 * 0.5) * gamma(nu2 * 0.5)))
  return (coef
      * (f ** (nu1 * 0.5 - 1.0))
      / ((1.0 + nu1 * f / nu2) ** ((nu1 + nu2) * 0.5)))

def buscar_f_critico(alpha, nu1, nu2):
  def objetivo(f_val):
    area, _ = integrate.quad(Fdist, f_val, np.inf, args=(nu1, nu2))
    return area - alpha
      
  res = root_scalar(objetivo, bracket=[1.0, 10.0], method='brentq')
  return res.root

### Problema 7

Para una distribuci´on F encuentre:
* f0.05 con v1 = 1 y v2 = 15.
* f0.05 con v1 = 15 y v2 = 7.
* f0.01 con v1 = 24 y v2 = 19
* f0.95 con v1 = 19 y v2 = 24.
* f0.99 con v1 = 28 y v2 = 12.

In [21]:
# Definición de la función de densidad F
def Fdist(f, nu1, nu2):
    coef = (gamma((nu1 + nu2) * 0.5)
            * ((nu1 / nu2) ** (nu1 * 0.5))
            / (gamma(nu1 * 0.5) * gamma(nu2 * 0.5)))
    return (coef
            * (f ** (nu1 * 0.5 - 1.0))
            / ((1.0 + nu1 * f / nu2) ** ((nu1 + nu2) * 0.5)))

# Función para encontrar el valor crítico
def buscar_f_critico(alpha, nu1, nu2):
    def objetivo(f_val):
        area, _ = integrate.quad(Fdist, f_val, np.inf, args=(nu1, nu2))
        return area - alpha
    
    # Se amplía el bracket para abarcar valores menores a 1 (como f_0.95 y f_0.99)
    res = root_scalar(objetivo, bracket=[0.001, 100.0], method='brentq')
    return res.root

# Casos solicitados del problema 7
f1 = buscar_f_critico(0.05, 1, 15)
f2 = buscar_f_critico(0.05, 15, 7)
f3 = buscar_f_critico(0.01, 24, 19)
f4 = buscar_f_critico(0.95, 19, 24)
f5 = buscar_f_critico(0.99, 28, 12)

# Impresión de resultados
print(f"f_0.05 con v1=1  y v2=15 = {f1:.4f}")
print(f"f_0.05 con v1=15 y v2=7  = {f2:.4f}")
print(f"f_0.01 con v1=24 y v2=19 = {f3:.4f}")
print(f"f_0.95 con v1=19 y v2=24 = {f4:.4f}")
print(f"f_0.99 con v1=28 y v2=12 = {f5:.4f}")

f_0.05 con v1=1  y v2=15 = 4.5431
f_0.05 con v1=15 y v2=7  = 3.5107
f_0.01 con v1=24 y v2=19 = 2.9249
f_0.95 con v1=19 y v2=24 = 0.4730
f_0.99 con v1=28 y v2=12 = 0.3453


### Problema 9
Si S2
1 y S2
2 son las varianzas de las variables aleatorias independientes de tama˜nos n1 = 10 y n2 = 15 de
poblaciones normales con varianzas iguales, encuentre P(S2
1/S2
2 ) < 4.03.

In [23]:
# 1. Definición de la distribución F de Fisher
def Fdist(f, nu1, nu2):
    # La distribución F solo está definida para valores > 0
    if f <= 0:
        return 0.0
        
    coef = (gamma((nu1 + nu2) * 0.5) 
            * ((nu1 / nu2) ** (nu1 * 0.5)) 
            / (gamma(nu1 * 0.5) * gamma(nu2 * 0.5)))
            
    termino = (f ** (nu1 * 0.5 - 1.0)) / ((1.0 + nu1 * f / nu2) ** ((nu1 + nu2) * 0.5))
    
    return coef * termino

# 2. Parámetros del problema 9
nu1 = 9   # Grados de libertad 1 (n1 - 1 = 10 - 1)
nu2 = 14  # Grados de libertad 2 (n2 - 1 = 15 - 1)
limite_superior = 4.03

# 3. Calcular la integral (Probabilidad acumulada desde 0 hasta 4.03)
probabilidad, _ = integrate.quad(Fdist, 0, limite_superior, args=(nu1, nu2))

print(f"Probabilidad P(S1^2 / S2^2 < 4.03) = {probabilidad:.4f} (o {probabilidad*100:.0f}%)")

Probabilidad P(S1^2 / S2^2 < 4.03) = 0.9900 (o 99%)


### Problema 12:
Se asegura que la resistencia del cable de dise˜no A es mayor que la resistencia del cable B. Un experimento
en los cables muestra los siguientes resultados (en ohms):
Page 2
Cable A Cable B
0.140 0.135
0.138 0.140
0.143 0.136
0.142 0.142
0.144 0.138
0.137 0.140
Asumiendo varianzas iguales, ¿qu´e conclusiones cuantitativas puede deducir a partir de los datos presentados?

In [25]:
def tdist(t, nu):
    coef = gamma((nu + 1) * 0.5) / (gamma(nu * 0.5) * np.sqrt(np.pi * nu))
    return coef * (1.0 + (t**2) / nu) ** (-(nu + 1.0) * 0.5)

# 2. Datos de los cables A y B
cable_a = np.array([0.140, 0.138, 0.143, 0.142, 0.144, 0.137])
cable_b = np.array([0.135, 0.140, 0.136, 0.142, 0.138, 0.140])

# 3. Tamaños y medias muestrales
n_a = len(cable_a)
n_b = len(cable_b)
media_a = np.mean(cable_a)
media_b = np.mean(cable_b)

# 4. Varianzas muestrales (ddof=1 para dividir entre n-1)
var_a = np.var(cable_a, ddof=1)
var_b = np.var(cable_b, ddof=1)

# 5. Varianza ponderada (Sp^2) y grados de libertad (nu)
nu = n_a + n_b - 2
var_p = ((n_a - 1) * var_a + (n_b - 1) * var_b) / nu
sp = np.sqrt(var_p)

# 6. Estadístico t calculado
t_stat = (media_a - media_b) / (sp * np.sqrt(1.0/n_a + 1.0/n_b))

# 7. Cálculo de la probabilidad (p-value para una cola derecha)
p_value, _ = integrate.quad(tdist, t_stat, np.inf, args=(nu,))

print(f"Media Cable A: {media_a:.5f}")
print(f"Media Cable B: {media_b:.5f}")
print(f"Estadístico t calculado: {t_stat:.3f}")
print(f"Grados de libertad: {nu}")
print(f"Valor p (Probabilidad): {p_value:.4f}")

# 8. Evaluación con nivel de significancia del 5%
alpha = 0.05
if p_value < alpha:
    print(f"\nConclusión: Como el valor p ({p_value:.4f}) es menor que {alpha}, SE RECHAZA H0.")
    print("Hay evidencia cuantitativa de que la resistencia de A es MAYOR que la de B.")
else:
    print(f"\nConclusión: Como el valor p ({p_value:.4f}) es mayor que {alpha}, NO SE RECHAZA H0.")
    print("No hay suficiente evidencia estadística al 5% para asegurar que la resistencia de A sea mayor que la de B.")

Media Cable A: 0.14067
Media Cable B: 0.13850
Estadístico t calculado: 1.372
Grados de libertad: 10
Valor p (Probabilidad): 0.1001

Conclusión: Como el valor p (0.1001) es mayor que 0.05, NO SE RECHAZA H0.
No hay suficiente evidencia estadística al 5% para asegurar que la resistencia de A sea mayor que la de B.


### Problema 14:
Una encuesta se realiza con la esperanza de comparar salarios de jefes de plantas qu´ımicas en dos ´areas
del pa´ıs, n la regi´on norte y en la oeste. Se toman muestras aleatorias independientes de tama˜no 300 de
cada regi´on. Se les pregunta a los jefes sus salarios anuales. Los resultados son como se sigue:
• Norte: x1 = 102, 300, s1 = 5700.
• Oeste: x2 = 98, 500, s2 = 3800.
(a) Construya un intervalo de confianza del 99% para μ1 −μ2, la diferencia entre los salarios promedio.
(b) ¿Qu´e asumi´o en el inciso anterior sobre la distribuci´on anual poblacional de los salarios en las
regiones? ¿Por qu´e tom´o esta decisi´on?
(c) Qu´e asumi´o sobre la distribuci´on de varianzas? ¿Es razonable asumir que las varianzas son iguales?
Explique esto.

In [26]:
# 1. Datos del problema
n1 = 300
x_bar1 = 102300
s1 = 5700

n2 = 300
x_bar2 = 98500
s2 = 3800

# 2. Diferencia puntual de las medias
diff_medias = x_bar1 - x_bar2

# 3. Error estándar sin asumir varianzas iguales (aproximación para muestras grandes)
error_estandar = np.sqrt((s1**2 / n1) + (s2**2 / n2))

# 4. Valor crítico Z para 99% de confianza (aprox 2.5758)
z_critico = 2.5758

# 5. Margen de error y límites
margen_error = z_critico * error_estandar
limite_inferior = diff_medias - margen_error
limite_superior = diff_medias + margen_error

print(f"Diferencia de salarios promedio: ${diff_medias:.2f}")
print(f"Error estándar: ${error_estandar:.2f}")
print(f"Intervalo de confianza del 99%: [${limite_inferior:.2f}, ${limite_superior:.2f}]")

Diferencia de salarios promedio: $3800.00
Error estándar: $395.52
Intervalo de confianza del 99%: [$2781.23, $4818.77]
